![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [18]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

#Replacing missing values
new_cc_apps = cc_apps.replace(np.nan)
cc_apps_imputed = new_cc_apps.copy()

# Iterate through each column
for col in cc_apps_imputed.columns:
    if cc_apps_imputed[col].dtype == 'object':
        # Fill with most frequent value (mode)
        cc_apps_imputed[col].fillna(cc_apps_imputed[col].value_counts().idxmax(), inplace=True)
    else:
        # Fill with mean for numeric columns
        cc_apps_imputed[col].fillna(cc_apps_imputed[col].mean(), inplace=True)

# Apply one-hot encoding
cc_apps_final = pd.get_dummies(cc_apps_imputed, drop_first=True)

# Define Features and Target Variable
X = cc_apps_final.iloc[:, :-1].values
y = cc_apps_final.iloc[:, -1].values

# Perform Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Logistic Regression
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)
y_pred = log_reg.predict(X_test_scaled)

#Confusion Matrix
print(confusion_matrix(y_test,y_pred))

#Defining Grid Search Parameters
param_grid = {
    "C": [0.01, 0.001, 0.0001],  # Regularization strength
    "max_iter": [100, 150, 200],   # Maximum iterations
}

#Performing Grid Search
grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

#Getting the best model
best_log_reg = grid_search.best_estimator_
print("Best parameters found:", grid_search.best_params_)
best_score = log_reg.score(X_test_scaled, y_test)
print("Model Accuracy:", best_score)

[[72 25]
 [18 92]]
Best parameters found: {'C': 0.01, 'max_iter': 100}
Model Accuracy: 0.7922705314009661
